# Advanced Certification Program in Computational Data Science

##  A program by IISc and TalentSprint

### Mini Project Notebook: Video based Action Classification using LSTM

## Learning Objectives

At the end of the experiment, you will be able to :

* extract frames out of a video
* build the CNN model to extract features from the video frames
* train LSTM/GRU model to perform action classification 

## Information

**Background:** The CNN LSTM architecture involves using Convolutional Neural Network (CNN) layers for feature extraction on input data combined with LSTMs to support sequence prediction.

CNN LSTMs were developed for visual time series prediction problems and the application of generating textual descriptions from sequences of images (e.g. videos). Specifically, the problems of:

 

*   Activity Recognition: Generating a textual description of an activity demonstrated in a sequence of images
*   Image Description: Generating a textual description of a single image.
*   Video Description: Generating a textual description of a sequence of images.

**Applications:** Applications such as surveillance, video retrieval and
human-computer interaction require methods for recognizing human actions in various scenarios. In the area of robotics, the tasks of
autonomous navigation or social interaction could also take advantage of the knowledge extracted
from live video recordings. Typical scenarios
include scenes with cluttered, moving backgrounds, nonstationary camera, scale variations, individual variations in
appearance and cloth of people, changes in light and view
point and so forth. All of these conditions introduce challenging problems that can be addressed using deep learning (computer vision) models.

## Dataset



**Dataset:** This dataset consists of labelled videos of 6 human actions (walking, jogging, running, boxing, hand waving and hand clapping) performed several times by 25 subjects in four different scenarios: outdoors s1, outdoors with scale variation s2, outdoors with different clothes s3 and indoors s4 as illustrated below. 

![img](https://cdn.iisc.talentsprint.com/CDS/Images/actions.gif)

All sequences were taken over homogeneous backgrounds with a static camera with 25fps frame rate. The sequences were downsampled to the spatial resolution of 160x120 pixels and have a length of four seconds in average. In summary, there are 25x6x4=600 video files for each combination of 25 subjects, 6 actions and 4 scenarios. For this mini-project we have randomly selected 20% of the data as test set.

Dataset source: https://www.csc.kth.se/cvap/actions/

**Methodology:** 

When performing image classification, we input an image to our CNN; Obtain the predictions from the CNN; 
Choose the label with the largest corresponding probability


Since a video is just a series of image frames, in a video classification, we Loop over all frames in the video file; 
For each frame, pass the frame through the CNN; Classify each frame individually and independently of each other; Choose the label with the largest corresponding probability; 
Label the frame and write the output frame to disk

Refer this [Video Classification using Keras](https://medium.com/video-classification-using-keras-and-tensorflow/action-recognition-and-video-classification-using-keras-and-tensorflow-56badcbe5f77) for complete understanding and implementation example of video classification.

## Problem Statement

Train a CNN-LSTM based deep neural net to recognize the action being performed in a video

## Grading = 10 Points

In [ ]:
from psutil import virtual_memory
ram_gb = virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))

if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')

In [ ]:
#@title Download Dataset
!wget -qq https://cdn.iisc.talentsprint.com/CDS/MiniProjects/Actions.zip
!unzip -qq Actions.zip
print("Dataset downloaded successfully!!")

### Import required packages

In [ ]:
import keras
from keras import applications
from keras import optimizers
from keras.models import Sequential, Model 
from keras.layers import *
from keras.applications.vgg16 import VGG16
from keras.models import Model
from keras.layers import Dense, Input
from keras.layers.pooling import GlobalAveragePooling2D
from keras.layers.recurrent import LSTM
from keras.layers import Conv2D, BatchNormalization, MaxPool2D, GlobalMaxPool2D
from keras.layers import TimeDistributed, GRU, Dense, Dropout
from keras.layers import Conv2D, BatchNormalization, MaxPool2D, GlobalMaxPool2D
from tensorflow.keras.optimizers import Adam

import os, glob
import cv2
import numpy as np

### Load the data and generate frames of video (2 points)

Detecting an action is possible by analyzing a series of images (that we name “frames”) that are taken in time.

Hint: Refer data preparation section in [keras_video_classification](https://keras.io/examples/vision/video_classification/)


In [ ]:
data_dir = "/content/Actions/train/"
test_data_dir = "/content/Actions/test/"

In [ ]:
!ls /content/Actions/train/*/*.avi

In [ ]:
listOfFiles=[]
for (dirpath, dirnames, filenames) in os.walk("/content/Actions/train/"):
    listOfFiles += [os.path.join(dirpath, file) for file in filenames]

#listOfFiles

j=0
for i in listOfFiles:
  print(i)
  j += 1

print(j)

In [ ]:
import pandas as pd
#https://github.com/mitu246/Activity-Recognition-in-Videos-using-Keras/blob/master/DL_Stretch_updated.ipynb
# Loading video names in a column and labels:
dict_gestures = {"handclapping":0,"walking":1,"boxing":2,"handwaving":3,"jogging":4,"running":5}
path='/content/Actions'

def videosFetchAll(cpath):
  listOfFiles = []
  label=[]
  c_directory=os.path.join(path,cpath)
  print(c_directory)
  c_directory = c_directory + "/" #"/*/" #+ "*.avi"
  print(c_directory)
  #c_videos=os.listdir(c_directory)
  #print(c_videos)

  for (dirpath, dirnames, filenames) in os.walk(c_directory):
    listOfFiles += [os.path.join(dirpath, file) for file in filenames]
  print(len(listOfFiles))
  print(listOfFiles)
  for temp in listOfFiles:
      i = temp.split("/")[-1].split(".")[0]
      print(i)
      if "handclapping" in i:
          label.append(dict_gestures["handclapping"])
      elif "walking" in i:
          label.append(dict_gestures["walking"])
      elif "boxing" in i:
          label.append(dict_gestures["boxing"])
      elif "handwaving" in i:
          label.append(dict_gestures["handwaving"])
      elif "jogging" in i:
          label.append(dict_gestures["jogging"])
      elif "running" in i:
          label.append(dict_gestures["running"])
      else:
          print("file_name_incorrect")

  print(len(label))
  print(label)
  videos=pd.DataFrame(listOfFiles,label).reset_index()
  videos.columns=["labels","video_name"]
  videos.groupby('labels').count()
  videos.head()
  return videos

In [ ]:
train_videos = videosFetchAll("train")
test_videos = videosFetchAll("test")

In [ ]:
train_videos.tail()

#### Visualize the frames and analyze the object in each frame. (1 point)

* Plot the frames of each class per row (6 rows)
* Plot the title as label on each subplot

In [ ]:
# YOUR CODE HERE

### Create the Neural Network (4 points)

We can build the model in several ways. We can use a well-known model that we inject in time distributed layer, or we can build our own.

With custom ConvNet each input image of the sequence must pass to a convolutional network. The goal is to train that model for each frame and then decide the class to infer.

* Use ConvNet and Time distributed to detect features.
* Inject the Time distributed output to GRU or LSTM to treat as a time series.
* Apply a DenseNet to take the decision and classify.

##### Build the ConvNet for the feature extraction, GRU LSTM layers as a time series and Dense layers for classification

In [ ]:
# Create Directories
stretches_dir_path=os.path.join(path,'Frames')
train_frames_dir=os.path.join(stretches_dir_path,"Train_Frames")
test_frames_dir=os.path.join(stretches_dir_path,"Test_Frames")
try:
    os.mkdir(stretches_dir_path)
except FileExistsError as ae:
    print("Folder Already Created")

try:
    os.mkdir(train_frames_dir)
except FileExistsError as ae:
    print("Folder Already Created")
    
try:
    os.mkdir(test_frames_dir)
except FileExistsError as ae:
    print("Folder Already Created")

In [ ]:
# Extract frames
import math
stretches_path=os.path.join(path,"Frames")
def video_capturing_function(dataset,folder_name):
    if "Train" in folder_name:
      video_directory = "/content/Actions/train/"
    else:
      video_directory = "/content/Actions/test/"
    print(video_directory)
    for i in np.arange(len(dataset)):
        video_name=dataset.video_name[i]
        print(video_name)
        video_read_path=os.path.join(video_directory,video_name)
        print(video_read_path)
        cap=cv2.VideoCapture(video_read_path)
        try:
            print(video_name.split("/")[-1].split(".")[0])
            print(os.path.join(os.path.join(stretches_path,folder_name),video_name.split("/")[-1].split(".")[0]))
            os.mkdir(os.path.join(os.path.join(stretches_path,folder_name),video_name.split("/")[-1].split(".")[0]))
        except:
            print("File Already Created")
        train_write_file=os.path.join(os.path.join(stretches_path,folder_name),video_name.split("/")[-1].split(".")[0])
        cap.set(cv2.CAP_PROP_FPS, 20)
        frameRate=cap.get(5)
        x=1
        count=0
        while(cap.isOpened()):
            frameId = cap.get(1) #current frame number
            ret, frame = cap.read()
            if (ret != True):
                break
            if (frameId % math.floor(frameRate) == 0):
                filename ="frame%d.jpg" % count;count+=1
                frame_grey=cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) 
                cv2.imwrite(os.path.join(train_write_file,filename), frame_grey)
                #cv2.imwrite(os.path.join(train_write_file,filename), frame) # vgg-16 expects 3 channels
        cap.release()
    return print("All frames written in the: "+folder_name+" Folder")
    
video_capturing_function(train_videos,"Train_Frames")
video_capturing_function(test_videos,"Test_Frames")

In [ ]:
# Data load function for 10 frames:
from keras.preprocessing import image
from PIL import Image
def data_load_function_10frames(dataset,directory):
    frames=[]
    for i in np.arange(len(dataset)):
        vid_name=dataset.video_name[i].split("/")[-1].split(".")[0]
        vid_dir_path=os.path.join(directory,vid_name)
        frames_to_select=[]
        for l in np.arange(0,7):
            frames_to_select.append('frame%d.jpg' % l)
        vid_data=[]
        for frame in frames_to_select:
            image=Image.open(os.path.join(vid_dir_path,frame))
            #image=image.resize((250, 250), Image.ANTIALIAS) 
            image=image.resize((224, 224), Image.ANTIALIAS) # VGG-16
            datu=np.asarray(image)
            normu_dat=datu/255
            vid_data.append(normu_dat)
        vid_data=np.array(vid_data)
        frames.append(vid_data)
    return np.array(frames)

In [ ]:
# 10 frames train, test data:

test_dataset_new=data_load_function_10frames(test_videos,"/content/Actions/Frames/Test_Frames")
train_dataset_new=data_load_function_10frames(train_videos,"/content/Actions/Frames/Train_Frames")

test_labels=np.array(test_videos.labels)
train_labels=np.array(train_videos.labels)

In [ ]:
print(train_dataset_new.shape)
print(train_labels.shape)

#train_dataset_new = train_dataset_new[0:25]
#train_labels = train_labels[0:25]
#vgg16
train_dataset_new = train_dataset_new[0:30]
train_labels = train_labels[0:30]

train_dataset_new.shape
train_labels.shape

#test_dataset_new = test_dataset_new[0:20]
#test_labels = test_labels[0:20]
#vgg-16
print(test_dataset_new.shape)
print(test_labels.shape)
test_dataset_new = test_dataset_new[0:20]
test_labels = test_labels[0:20]


print(train_dataset_new.shape)
print(train_labels.shape)
print(test_dataset_new.shape)
print(test_labels.shape)

In [ ]:
print(test_labels.shape)
test_dataset_new.shape

In [ ]:
# Reshaping tensors to conform with the model we are going to train:
test_dataset_new=test_dataset_new.reshape((20,7,224,224,1))
train_dataset_new=train_dataset_new.reshape((30,7,224,224,1))

#test_dataset_new=test_dataset_new.reshape((120,7,224,224,3))
#train_dataset_new=train_dataset_new.reshape((479,7,224,224,3)) # VGG-16
#train_dataset_new.shape

In [ ]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
lb_encoder = LabelEncoder()
train_labels = lb_encoder.fit_transform(train_labels)
train_labels = to_categorical(train_labels)

test_labels = lb_encoder.transform(test_labels)
test_labels = to_categorical(test_labels)

#### Setup the parameters and train the model with epochs, batch wise 

* Use train data to fit the model and test data for validation
* Configure batch size and epochs
* Plot the loss of train and test data

In [ ]:
# Note: There will be a high memory requirement for the training steps below. 
# You should work on a GPU/TPU based runtime. See 'Change Runtime' in Colab
# Training time for each epoch could be ~30 mins
# To save and re-load your model later, see the reference below:
# https://colab.research.google.com/github/tensorflow/docs/blob/master/site/en/tutorials/keras/save_and_load.ipynb

from keras.layers import TimeDistributed, Conv2D, Dense, MaxPooling2D, Flatten, LSTM, Dropout, BatchNormalization
from keras import models
model_cnlst = models.Sequential()
#model_cnlst.add(TimeDistributed(Conv2D(128, (3, 3), strides=(1,1),activation='relu'),input_shape=(7, 250, 250, 1)))
model_cnlst.add(TimeDistributed(Conv2D(64, (3, 3), strides=(1,1),activation='relu'),input_shape=(7, 224, 224, 1)))
#model_cnlst.add(TimeDistributed(Conv2D(64, (3, 3), strides=(1,1),activation='relu')))
model_cnlst.add(TimeDistributed(MaxPooling2D(2,2)))
#model_cnlst.add(TimeDistributed(Conv2D(64, (3, 3), strides=(1,1),activation='relu')))
model_cnlst.add(TimeDistributed(Conv2D(32, (3, 3), strides=(1,1),activation='relu')))
#model_cnlst.add(TimeDistributed(MaxPooling2D(2,2)))
model_cnlst.add(TimeDistributed(BatchNormalization()))


model_cnlst.add(TimeDistributed(Flatten()))
model_cnlst.add(Dropout(0.2))

model_cnlst.add(LSTM(32,return_sequences=False,dropout=0.2)) # used 32 units

model_cnlst.add(Dense(64,activation='relu'))
#model_cnlst.add(Dense(32,activation='relu'))
model_cnlst.add(Dropout(0.2))
model_cnlst.add(Dense(6, activation='sigmoid'))
model_cnlst.summary()

In [ ]:
# 'cnn_lstm_model_new5.h5'
from keras import optimizers
callbacks_list_cnlst=[keras.callbacks.EarlyStopping(
monitor='acc',patience=3),
               keras.callbacks.ModelCheckpoint(
               filepath='cnn_lstm_model_new3.h5',
               monitor='val_loss',
               save_best_only=True),
                keras.callbacks.ReduceLROnPlateau(monitor = "val_loss", factor = 0.1, patience = 3)
               ]

model_cnlst.compile(loss='sparse_categorical_crossentropy',metrics=["accuracy"])

In [ ]:
#https://github.com/mitu246/Activity-Recognition-in-Videos-using-Keras/blob/master/DL_Stretch_updated.ipynb
#history_new_cnlst=model_cnlst.fit(train_dataset_new,train_labels,epochs=1,callbacks=callbacks_list_cnlst)
history_new_cnlst=model_cnlst.fit(train_dataset_new,train_labels,epochs=12,batch_size=1)

In [ ]:
import matplotlib.pyplot as plt
acc=history_new_cnlst.history["accuracy"]
loss=history_new_cnlst.history["loss"]
epochs=np.arange(0,12)

plt.figure()
plt.subplot(2,2,1)
plt.plot(epochs,loss,'-o')
plt.title('Training Loss')

plt.subplot(2,2,2)
plt.plot(epochs,acc,'-o')
plt.title('Train Accuracy')

plt.tight_layout()

In [ ]:
#modelu_5=load_model('cnn_lstm_model_new3.h5')
model_cnlst.evaluate(test_dataset_new,test_labels)

In [ ]:
model_cnlst = models.Sequential()
model_cnlst.add(TimeDistributed(Conv2D(128, (3, 3), strides=(1,1),activation='relu'),input_shape=(7, 224, 224, 1)))
model_cnlst.add(TimeDistributed(Conv2D(64, (3, 3), strides=(1,1),activation='relu')))
model_cnlst.add(TimeDistributed(MaxPooling2D(2,2)))
model_cnlst.add(TimeDistributed(Conv2D(64, (3, 3), strides=(1,1),activation='relu')))
model_cnlst.add(TimeDistributed(Conv2D(32, (3, 3), strides=(1,1),activation='relu')))
model_cnlst.add(TimeDistributed(BatchNormalization()))
model_cnlst.add(TimeDistributed(MaxPooling2D(2,2)))

model_cnlst.add(TimeDistributed(Conv2D(64, (3, 3), strides=(1,1),activation='relu')))
model_cnlst.add(TimeDistributed(Conv2D(32, (3, 3), strides=(1,1),activation='relu')))
model_cnlst.add(TimeDistributed(MaxPooling2D(2,2)))
model_cnlst.add(TimeDistributed(Conv2D(64, (3, 3), strides=(1,1),activation='relu')))
model_cnlst.add(TimeDistributed(Conv2D(32, (3, 3), strides=(1,1),activation='relu')))
model_cnlst.add(TimeDistributed(MaxPooling2D(2,2)))
model_cnlst.add(TimeDistributed(BatchNormalization()))
model_cnlst.add(TimeDistributed(Flatten()))
model_cnlst.add(Dropout(0.2))

model_cnlst.add(LSTM(64,return_sequences=False,dropout=0.2)) # used 32 units
model_cnlst.add(Dense(128,activation='relu'))
model_cnlst.add(BatchNormalization())
model_cnlst.add(Dense(64,activation='relu'))
model_cnlst.add(Dense(32,activation='relu'))
model_cnlst.add(Dropout(0.2))
model_cnlst.add(Dense(6, activation='sigmoid'))
model_cnlst.summary()

In [ ]:
callbacks_list_cnlst=[keras.callbacks.EarlyStopping(
monitor='acc',patience=3),
               keras.callbacks.ModelCheckpoint(
               filepath='cnn_lstm_model_new4.h5',
               monitor='val_loss',
               save_best_only=True),
               keras.callbacks.ReduceLROnPlateau(monitor = "val_loss", factor = 0.1, patience = 3)
               ]

from keras import optimizers
optimizer_new=optimizers.rmsprop_v2
model_cnlst.compile(loss='sparse_categorical_crossentropy',metrics=["accuracy"])


In [ ]:
history=model_cnlst.fit(train_dataset_new,train_labels,batch_size=1,epochs=12)

In [ ]:
acc=history.history["accuracy"]
loss=history.history["loss"]
epochs=np.arange(0,12)

In [ ]:
plt.figure()
plt.subplot(2,2,1)
plt.plot(epochs,loss,'-o')
plt.title('Training Loss')

plt.subplot(2,2,2)
plt.plot(epochs,acc,'-o')
plt.title('Train Accuracy')

plt.tight_layout()

In [ ]:
#from keras.models import load_model
#modelu_5=load_model('cnn_lstm_model_new4.h5')
model_cnlst.evaluate(test_dataset_new,test_labels)

# **3D CNN**

In [ ]:
from keras.layers import Conv3D, MaxPooling3D, BatchNormalization, Dropout, Dense, Flatten, concatenate
from keras.models import Model
from keras import Input
# 3D Convolutional Model:
input_model=Input(shape=(7,224,224,1))
layer=Conv3D(32,(3,3,3),strides=(1,1,1),activation='relu')(input_model)
layer=MaxPooling3D((2,2,2))(layer)
layer=MaxPooling3D((2,2,2))(layer)
layer=BatchNormalization()(layer)
layer=Flatten()(layer)
layer=Dense(128,activation='relu')(layer)
layer=Dropout(0.1)(layer)
layer=Dense(64,activation='relu')(layer)
layer=Dense(32,activation='relu')(layer)
layer_output=Dense(6,activation='sigmoid')(layer)

model_3dConv=Model(input_model,layer_output)

model_3dConv.summary()

In [ ]:
# Conv3d model training:
from keras import optimizers
#optimizer_new=optimizers.RMSprop(lr=0.1)
#optimizer_adagrad=keras.optimizers.adagrad_v2
callbacks_list_conv_3d=[keras.callbacks.EarlyStopping(
monitor='acc',patience=6),
               keras.callbacks.ModelCheckpoint(
               filepath='stretch_model_conv_3d_new4.h5',
               monitor='val_loss',
               save_best_only=True),
                        keras.callbacks.ReduceLROnPlateau(monitor = "val_loss", factor = 0.1, patience = 2)
               ]
model_3dConv.compile(loss='sparse_categorical_crossentropy',metrics=["accuracy"])
conv_3d_model_history=model_3dConv.fit(train_dataset_new,train_labels,batch_size=1,epochs=12)

In [ ]:
import matplotlib.pyplot as plt
acc=conv_3d_model_history.history["accuracy"]
loss=conv_3d_model_history.history["loss"]

epochs=np.arange(0,12)

plt.figure()
plt.subplot(2,2,1)
plt.plot(epochs,loss,'-o')
plt.title('Training Loss')

plt.subplot(2,2,2)
plt.plot(epochs,acc,'-o')
plt.title('Train Accuracy')

plt.tight_layout()

In [ ]:
# modelu_6=load_model('stretch_model_conv_3d_new3.h5')
#modelu_6=load_model('stretch_model_conv_3d_new4.h5')

model_3dConv.evaluate(test_dataset_new,test_labels)

In [ ]:
test_preds=model_3dConv.predict(test_dataset_new)

In [ ]:
#https://github.com/mitu246/Activity-Recognition-in-Videos-using-Keras/blob/master/DL_Stretch_updated.ipynb
# Check for ensemble of 3 models

### Use pre-trained model for feature extraction (3 points)

To create a deep learning network for video classification:

* Convert videos to sequences of feature vectors using a pretrained convolutional neural network, such as VGG16, to extract features from each frame.

* Train an LSTM network on the sequences to predict the video labels.

* Assemble a network that classifies videos directly by combining layers from both networks.

Hint: [VGG-16 CNN and LSTM](https://riptutorial.com/keras/example/29812/vgg-16-cnn-and-lstm-for-video-classification)

#### Load and fine-tune the pre-trained model

In [ ]:
#https://riptutorial.com/keras/example/29812/vgg-16-cnn-and-lstm-for-video-classification
from keras.applications.vgg16 import VGG16
from keras.models import Model
from keras.layers import Dense, Input
from keras.layers.pooling import GlobalAveragePooling2D
from keras.layers.recurrent import LSTM
from keras.layers.wrappers import TimeDistributed

video = Input(shape=(7,
                     224,
                     224,
                     3))
cnn_base = VGG16(input_shape=(224,
                              224,3),
                 weights="imagenet",
                 include_top=False)

cnn_out = GlobalAveragePooling2D()(cnn_base.output)
cnn = Model(cnn_base.input, cnn_out)
cnn.trainable = False
encoded_frames = TimeDistributed(cnn)(video)
encoded_sequence = LSTM(256)(encoded_frames)
hidden_layer = Dense(units=6, activation="relu")(encoded_sequence)
outputs = Dense(units=6, activation="softmax")(hidden_layer)
modelvgg = Model([video], outputs)
modelvgg.compile(loss="sparse_categorical_crossentropy",
              metrics=["accuracy"]) 
#modelvgg.compile() 

#### Setup the parameters and train the model with epochs, batch wise

* Use train data to fit the model and test data for validation
* Configure batch size and epochs
* Plot the loss of train and test data

In [ ]:
modelvgg_history=modelvgg.fit(train_dataset_new,train_labels,batch_size=1,epochs=12)

In [ ]:
modelvgg_history.history

In [ ]:
import matplotlib.pyplot as plt
acc=modelvgg_history.history["accuracy"]
loss=modelvgg_history.history["loss"]

epochs=np.arange(0,12)

plt.figure()
plt.subplot(2,2,1)
plt.plot(epochs,loss,'-o')
plt.title('Training Loss')

plt.subplot(2,2,2)
plt.plot(epochs,acc,'-o')
plt.title('Train Accuracy')

plt.tight_layout()

In [ ]:
modelvgg.evaluate(test_dataset_new,test_labels)

In [ ]:
test_preds=modelvgg.predict(test_dataset_new)


In [ ]:
test_preds

### Report Analysis

* Discuss on FPS, Number of frames and duration of each video
* Analyze the impact of the LSTM, GRU and TimeDistributed layers
* Discuss about the model convergence using pre-trained and ConvNet
* *Additional Reading*: Read and discuss about the use of Conv3D in video classification